# Database Extractor Feedback Results (5 Application Case Studies)

This notebook runs three practical application scenarios using `DatabaseExtractorFeature` and summarizes the extraction feedback in a CSV-like table.

In [2]:
import csv
import json
import os
import sys
import time
from pathlib import Path

# Walk UP until we find the EMOS repo root (identified by both Features/ AND Information_Units/).
workspace_root = Path(os.path.abspath('')).resolve()
while workspace_root != workspace_root.parent:
    if (workspace_root / 'Features').is_dir() and (workspace_root / 'Information_Units').is_dir():
        break
    workspace_root = workspace_root.parent

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from Features.Materials_Exploration.DatabaseExtractor.DatabaseExtractorFeature import DatabaseExtractorFeature

# CIFs are kept in memory only — nothing is written to disk during extraction.
# Call save_cifs(case_slug) at any time to reproduce the files on disk.
cif_store: dict = {}

OUTPUT_ROOT = workspace_root / 'Features' / 'Materials_Exploration' / 'DatabaseExtractor' / 'analysis'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def _flatten_cifs(extraction):
    cifs = []
    for db_key, db_entry in extraction.get('databases', {}).items():
        payload = db_entry.get('payload', {})
        for cif in (payload.get('cif_strings', []) if isinstance(payload, dict) else []):
            cifs.append((db_key, cif))
    return cifs


def save_cifs(case_slug, limit=300):
    """Write CIFs for a case from cif_store to disk (for reproducibility).

    Parameters
    ----------
    case_slug : str
        Key used in cif_store (e.g. 'wide_bandgap').
    limit : int
        Maximum number of files to write (default 300).
    """
    records = cif_store.get(case_slug, [])
    if not records:
        print(f"No CIFs in memory for '{case_slug}'. Re-run the extraction cell first.")
        return
    case_dir = OUTPUT_ROOT / case_slug
    case_dir.mkdir(parents=True, exist_ok=True)
    for i, (db_name, cif_text) in enumerate(records[:limit], start=1):
        (case_dir / f"{i:03d}_{db_name}.cif").write_text(cif_text)
    print(f"Saved {min(len(records), limit)} CIF files to: {case_dir}")


def run_extraction(case_name, case_slug, payload):
    """Run a single extraction. CIFs are stored in cif_store[case_slug] (in memory)."""
    feature = DatabaseExtractorFeature()
    inputs = feature.extract_inputs(payload)
    qv = payload.get('queryValues', payload.get('query_values', {}))
    inputs['selected_properties'] = list(qv.keys())

    started = time.time()
    extraction = feature.process_feature(inputs)
    elapsed = round(time.time() - started, 2)

    cif_records = _flatten_cifs(extraction)
    cif_store[case_slug] = cif_records          # keep in memory, not on disk

    dbs = ' + '.join(
        d.get('name', d.get('value', '?')) if isinstance(d, dict) else str(d)
        for d in payload.get('active_databases', [])
    )
    props = '; '.join(
        f"{k}: {v[0]} to {v[1]}" if isinstance(v, list) and len(v) == 2 else f"{k}: {v}"
        for k, v in qv.items()
    )
    row = {
        'Application Scenario': case_name,
        'Chosen Databases': dbs,
        'Target Compositions': payload.get('targetCompositions', payload.get('target_compositions', '')),
        'Chosen Properties': props,
        'Retrieval Mode': payload.get('retrievalMode', payload.get('retrieval_mode', 'lenient')),
        'Number of Candidate Materials Extracted': len(cif_records),
        'Time Consumed (s)': elapsed,
    }
    print(json.dumps(row, indent=2))
    return row


print(f"Setup complete. Workspace root: {workspace_root}")
print(f"Analysis directory: {OUTPUT_ROOT}")
print("CIFs are held in memory. Call save_cifs('<case_slug>') to write them to disk.")


Setup complete. Workspace root: /home/soe/EMOS
Analysis directory: /home/soe/EMOS/Features/Materials_Exploration/DatabaseExtractor/analysis
CIFs are held in memory. Call save_cifs('<case_slug>') to write them to disk.


In [3]:
# Application 1: Wide-bandgap materials for power electronics
# Target: AlN | Properties: band_gap [2.0–6.5 eV], nelements [2]
# Databases: AFLOW + JARVIS-DFT + Alexandria | lenient
# Alexandria also maps band_gap (_alexandria_band_gap) and nelements, so all three
# databases contribute results in lenient mode.

row_1 = run_extraction(
    case_name='Wide-bandgap materials discovery',
    case_slug='wide_bandgap',
    payload={
        'batchSize': 120,
        'retrievalMode': 'lenient',
        'targetCompositions': 'AlN',
        'queryValues': {
            'band_gap': [2.0, 6.5],
            'nelements': [2, 2],
        },
        'active_databases': [
            {'value': 'aflow',     'name': 'AFLOW'},
            {'value': 'jarvisdft', 'name': 'JARVIS-DFT'},
            {'value': 'alexandria','name': 'Alexandria'},
        ],
    },
)


/home/soe/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 8 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/home/soe/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 6 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


{
  "Application Scenario": "Wide-bandgap materials discovery",
  "Chosen Databases": "AFLOW + JARVIS-DFT + Alexandria",
  "Target Compositions": "AlN",
  "Chosen Properties": "band_gap: 2.0 to 6.5; nelements: 2 to 2",
  "Retrieval Mode": "lenient",
  "Number of Candidate Materials Extracted": 44,
  "Time Consumed (s)": 83.29
}


In [4]:
# Application 2: Semiconductor channel material pre-screening
# Target: GaAs | Properties: band_gap [0.8–2.2 eV], hull_distance [0–0.2 eV/atom] | JARVIS-DFT + Alexandria | lenient
# Probed: returns ~28 CIF files.

row_2 = run_extraction(
    case_name='Semiconductor channel material pre-screening',
    case_slug='semiconductor_channel',
    payload={
        'batchSize': 120,
        'retrievalMode': 'lenient',
        'targetCompositions': 'GaAs',
        'queryValues': {
            'band_gap': [0.8, 2.2],
            'hull_distance': [0.0, 0.2],
        },
        'active_databases': [
            {'value': 'jarvisdft', 'name': 'JARVIS-DFT'},
            {'value': 'alexandria', 'name': 'Alexandria'},
        ],
    },
)

{
  "Application Scenario": "Semiconductor channel material pre-screening",
  "Chosen Databases": "JARVIS-DFT + Alexandria",
  "Target Compositions": "GaAs",
  "Chosen Properties": "band_gap: 0.8 to 2.2; hull_distance: 0.0 to 0.2",
  "Retrieval Mode": "lenient",
  "Number of Candidate Materials Extracted": 28,
  "Time Consumed (s)": 21.04
}


In [5]:
# Application 3: Thermal/mechanical materials shortlist
# Target: SiC | Properties: band_gap [2.0–8.0 eV], bulk_modulus [100–500 GPa]
# Databases: AFLOW + JARVIS-DFT + MatHub3D + Alexandria | lenient
# AFLOW, JARVIS-DFT, and MatHub3D all map both band_gap and bulk_modulus with range support.
# Alexandria maps band_gap only — in lenient mode it contributes that filter.

row_3 = run_extraction(
    case_name='Thermal/mechanical materials shortlist',
    case_slug='thermal_mechanical',
    payload={
        'batchSize': 120,
        'retrievalMode': 'lenient',
        'targetCompositions': 'SiC',
        'queryValues': {
            'band_gap':    [2.0, 8.0],
            'bulk_modulus': [100, 500],
        },
        'active_databases': [
            {'value': 'aflow',     'name': 'AFLOW'},
            {'value': 'jarvisdft', 'name': 'JARVIS-DFT'},
            {'value': 'mathub3d',  'name': 'MatHub3D'},
            {'value': 'alexandria','name': 'Alexandria'},
        ],
    },
)


/home/soe/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 16 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/home/soe/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 28 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/home/soe/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/io/cif.py:1314: UserWarning: Some occupancies ([2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0]) sum to > 1! If they are within the occupancy_tolerance, they will be rescaled. The current occupancy_tolerance is set to: 1.0
  if struct := self._get_structure(data, primitive, symmetrized, check_occu=check_occu):
/home/soe/EMOS/emos_env/

{
  "Application Scenario": "Thermal/mechanical materials shortlist",
  "Chosen Databases": "AFLOW + JARVIS-DFT + MatHub3D + Alexandria",
  "Target Compositions": "SiC",
  "Chosen Properties": "band_gap: 2.0 to 8.0; bulk_modulus: 100 to 500",
  "Retrieval Mode": "lenient",
  "Number of Candidate Materials Extracted": 96,
  "Time Consumed (s)": 118.31
}


In [6]:

# ── Case 4: Photocatalytic Oxide Screening — Strict Cross-DB Filtering ───────
# Reverse-engineered strict-safe pair: AFLOW maps band_gap → Egap (range OK)
# and nelements → nspecies (range OK); JARVIS maps identically via OPTIMADE.
# These two properties are the intersection that guarantees BOTH databases
# survive strict mode without being dropped.
# Target: TiO2 — rutile/anatase/brookite polymorphs, benchmark photocatalyst.

payload_4 = {
    'batchSize': 150,
    'retrievalMode': 'strict',
    'targetCompositions': 'TiO2',
    'queryValues': {
        'band_gap':  [0.0, 6.0],
        'nelements': [2, 2],
    },
    'active_databases': [
        {'value': 'aflow',    'name': 'AFLOW'},
        {'value': 'jarvisdft','name': 'JARVIS-DFT'},
    ],
}

row_4 = run_extraction(
    case_name='Photocatalytic oxide screening (strict)',
    case_slug='photocatalyst_strict',
    payload=payload_4,
)
print(json.dumps(row_4, indent=2))


/home/soe/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


{
  "Application Scenario": "Photocatalytic oxide screening (strict)",
  "Chosen Databases": "AFLOW + JARVIS-DFT",
  "Target Compositions": "TiO2",
  "Chosen Properties": "band_gap: 0.0 to 6.0; nelements: 2 to 2",
  "Retrieval Mode": "strict",
  "Number of Candidate Materials Extracted": 152,
  "Time Consumed (s)": 159.64
}
{
  "Application Scenario": "Photocatalytic oxide screening (strict)",
  "Chosen Databases": "AFLOW + JARVIS-DFT",
  "Target Compositions": "TiO2",
  "Chosen Properties": "band_gap: 0.0 to 6.0; nelements: 2 to 2",
  "Retrieval Mode": "strict",
  "Number of Candidate Materials Extracted": 152,
  "Time Consumed (s)": 159.64
}


In [7]:

# ── Case 5: UV Photodetector Semiconductor — Strict Cross-DB Filtering ───────
# Reverse-engineered strict-safe pair: JARVIS maps band_gap →
# _jarvis_optb88vdw_bandgap and formation_energy_per_atom →
# _jarvis_formation_energy_peratom (both range OK). Alexandria maps identically
# via OPTIMADE. These are the only two energetic/electronic properties
# simultaneously range-queryable in BOTH databases, so both survive strict mode.
# Target: ZnO — wurtzite/rocksalt/zincblende polymorphs, UV photodetector.

payload_5 = {
    'batchSize': 150,
    'retrievalMode': 'strict',
    'targetCompositions': 'ZnO',
    'queryValues': {
        'band_gap':                  [0.0, 5.0],
        'formation_energy_per_atom': [-5.0, 0.0],
    },
    'active_databases': [
        {'value': 'jarvisdft', 'name': 'JARVIS-DFT'},
        {'value': 'alexandria', 'name': 'Alexandria'},
    ],
}

row_5 = run_extraction(
    case_name='UV photodetector semiconductor (strict)',
    case_slug='photodetector_strict',
    payload=payload_5,
)
print(json.dumps(row_5, indent=2))


{
  "Application Scenario": "UV photodetector semiconductor (strict)",
  "Chosen Databases": "JARVIS-DFT + Alexandria",
  "Target Compositions": "ZnO",
  "Chosen Properties": "band_gap: 0.0 to 5.0; formation_energy_per_atom: -5.0 to 0.0",
  "Retrieval Mode": "strict",
  "Number of Candidate Materials Extracted": 160,
  "Time Consumed (s)": 19.81
}
{
  "Application Scenario": "UV photodetector semiconductor (strict)",
  "Chosen Databases": "JARVIS-DFT + Alexandria",
  "Target Compositions": "ZnO",
  "Chosen Properties": "band_gap: 0.0 to 5.0; formation_energy_per_atom: -5.0 to 0.0",
  "Retrieval Mode": "strict",
  "Number of Candidate Materials Extracted": 160,
  "Time Consumed (s)": 19.81
}


In [27]:

# Build and export a CSV-like summary table for all five application scenarios.
rows = [row_1, row_2, row_3, row_4, row_5]

headers = [
    'Application Scenario',
    'Chosen Databases',
    'Target Compositions',
    'Chosen Properties',
    'Retrieval Mode',
    'Number of Candidate Materials Extracted',
    'Time Consumed (s)',
]

summary_csv_path = OUTPUT_ROOT / 'emos_data_extraction_feedback_results.csv'
with summary_csv_path.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=headers, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(rows)

try:
    import pandas as pd
    df = pd.DataFrame(rows, columns=headers)
    display(df)
except Exception:
    print(json.dumps(rows, indent=2))

print(f'\nSummary CSV saved to: {summary_csv_path}')
print(f"\nIn-memory CIF store keys: {list(cif_store.keys())}")
print("To save any case's CIFs to disk, call: save_cifs('<case_slug>')")


,Application Scenario,Chosen Databases,Target Compositions,Chosen Properties,Retrieval Mode,Number of Candidate Materials Extracted,Time Consumed (s)
0,Wide-bandgap materials discovery,AFLOW + JARVIS-DFT + Alexandria,AlN,band_gap: 2.0 to 6.5; nelements: 2 to 2,lenient,44,62.12
1,Semiconductor channel material pre-screening,JARVIS-DFT + Alexandria,GaAs,band_gap: 0.8 to 2.2; hull_distance: 0.0 to 0.2,lenient,28,17.98
2,Thermal/mechanical materials shortlist,AFLOW + JARVIS-DFT + MatHub3D + Alexandria,SiC,band_gap: 2.0 to 8.0; bulk_modulus: 100 to 500,lenient,96,125.37
3,Photocatalytic oxide screening (strict),AFLOW + JARVIS-DFT,TiO2,band_gap: 0.0 to 6.0; nelements: 2 to 2,strict,152,160.56
4,UV photodetector semiconductor (strict),JARVIS-DFT + Alexandria,ZnO,band_gap: 0.0 to 5.0; formation_energy_per_ato...,strict,160,19.49



Summary CSV saved to: /home/soe/EMOS/Features/Materials_Exploration/DatabaseExtractor/results/emos_data_extraction_feedback_results.csv

In-memory CIF store keys: ['wide_bandgap', 'semiconductor_channel', 'thermal_mechanical', 'photocatalyst_strict', 'photodetector_strict']
To save any case's CIFs to disk, call: save_cifs('<case_slug>')


In [8]:
# Save all 5 case CIF files to /tmp/emos_cif_files/ for sharing.
TMP_ROOT = Path('/tmp/emos_cif_files')
TMP_ROOT.mkdir(parents=True, exist_ok=True)

for slug, records in cif_store.items():
    case_dir = TMP_ROOT / slug
    case_dir.mkdir(parents=True, exist_ok=True)
    for i, (db_name, cif_text) in enumerate(records[:300], start=1):
        (case_dir / f"{i:03d}_{db_name}.cif").write_text(cif_text)
    print(f"  {slug}: {min(len(records), 300)} CIF files → {case_dir}")

print(f"\nAll CIFs written to: {TMP_ROOT}")


  wide_bandgap: 44 CIF files → /tmp/emos_cif_files/wide_bandgap
  semiconductor_channel: 28 CIF files → /tmp/emos_cif_files/semiconductor_channel
  thermal_mechanical: 96 CIF files → /tmp/emos_cif_files/thermal_mechanical
  photocatalyst_strict: 152 CIF files → /tmp/emos_cif_files/photocatalyst_strict
  photodetector_strict: 160 CIF files → /tmp/emos_cif_files/photodetector_strict

All CIFs written to: /tmp/emos_cif_files
